# Part 4: nn.Module, layers and activation functions

In [1]:
import torch
from torch import nn

torch.manual_seed(0)
layer = nn.Linear(3, 2)
print("weight shape:", layer.weight.shape, " bias shape:", layer.bias.shape)
x = torch.randn(5, 3)              # a batch of five rows with three features each
print("input:", x.shape, " output:", layer(x).shape)

weight shape: torch.Size([2, 3])  bias shape: torch.Size([2])
input: torch.Size([5, 3])  output: torch.Size([5, 2])


## A linear layer is a matrix product plus a bias

In [2]:
by_hand = x @ layer.weight.T + layer.bias
print("same as the layer:", torch.allclose(by_hand, layer(x)))

same as the layer: True


## Activation functions

Without an activation, two linear layers collapse into one. The activation is what lets depth add power.

In [3]:
v = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
print("input  :", v)
print("ReLU   :", torch.relu(v))
print("sigmoid:", torch.sigmoid(v).round(decimals=3))
print("tanh   :", torch.tanh(v).round(decimals=3))

input  : tensor([-2.0000, -0.5000,  0.0000,  0.5000,  2.0000])
ReLU   : tensor([0.0000, 0.0000, 0.0000, 0.5000, 2.0000])
sigmoid: tensor([0.1190, 0.3780, 0.5000, 0.6220, 0.8810])
tanh   : tensor([-0.9640, -0.4620,  0.0000,  0.4620,  0.9640])


## A model as a class

In [4]:
from tiny_net import TinyNet

model = TinyNet()
print(model)
x = torch.randn(5, 3)
print("output shape:", model(x).shape)

TinyNet(
  (hidden): Linear(in_features=3, out_features=4, bias=True)
  (activation): ReLU()
  (out): Linear(in_features=4, out_features=1, bias=True)
)
output shape: torch.Size([5, 1])


## Parameters

Every weight and bias registered inside the module is a parameter, and the optimizer will receive all of them with one call.

In [5]:
for name, p in model.named_parameters():
    print(f"{name:16s} {tuple(p.shape)}  requires_grad={p.requires_grad}")
total = sum(p.numel() for p in model.parameters())
print("total parameters:", total, " = (3*4 + 4) + (4*1 + 1)")
print("state_dict keys:", list(model.state_dict().keys()))

hidden.weight    (4, 3)  requires_grad=True
hidden.bias      (4,)  requires_grad=True
out.weight       (1, 4)  requires_grad=True
out.bias         (1,)  requires_grad=True
total parameters: 21  = (3*4 + 4) + (4*1 + 1)
state_dict keys: ['hidden.weight', 'hidden.bias', 'out.weight', 'out.bias']


## The same network with Sequential

In [6]:
same = nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4, 1))
print(same)
print("parameters:", sum(p.numel() for p in same.parameters()))

Sequential(
  (0): Linear(in_features=3, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=1, bias=True)
)
parameters: 21


## Training mode and evaluation mode

Some layers behave differently while training. Dropout switches units off at random during training only. Call `model.eval()` before you predict.

In [7]:
drop = nn.Dropout(p=0.5)
ones = torch.ones(8)
torch.manual_seed(1)
drop.train()
print("training:  ", drop(ones))
drop.eval()
print("evaluation:", drop(ones))

training:   tensor([2., 2., 2., 2., 0., 2., 2., 0.])
evaluation: tensor([1., 1., 1., 1., 1., 1., 1., 1.])


## Your turn

1. Add a second hidden layer of four units to `TinyNet` and print the new parameter count.
2. Predict how many parameters `nn.Linear(9, 16)` has, then check with `sum(p.numel() for p in layer.parameters())`.
3. Remove the `ReLU` from `TinyNet.forward` and describe why this network can now only learn straight-line relationships.